# COVID-19 Data Analysis Project

## Comprehensive Analysis: Loading, ETL, Modeling, and Forecasting

---

# Урок 37: Завантаження, очищення, перші візуалізації
## Lesson 37: Loading, Cleaning, Initial Visualizations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
import plotly.express as px
import plotly.graph_objects as go

%matplotlib inline

## 1.1 Data Loading

In [ ]:
data_url = 'https://ourworldindata.org/explorers/covid'
data = 'covid_data.csv'
df = pd.read_csv(data)
df

## 1.2 Data Overview

In [ ]:
df.info()

In [ ]:
df['continent'].value_counts()

In [ ]:
df.columns

## 1.3 Data Quality Check

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().mean()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

## 1.4 Initial Visualizations

In [ ]:
df['date'] = pd.to_datetime(df['date'])

country = 'Ukraine'
df_country = df[df['location'] == country]

plt.figure(figsize=(12, 6))
plt.plot(df_country['date'], df_country['total_cases'], label='Total Cases')
plt.plot(df_country['date'], df_country['total_deaths'], label='Total Deaths')
plt.title(f'COVID-19 in country {country}')
plt.xlabel('Date')
plt.ylabel('Count')
plt.legend()
plt.show()

In [ ]:
country = 'Mexico'
df_country = df[df['location'] == country]

plt.figure(figsize=(12, 6))
plt.plot(df_country['date'], df_country['total_cases'], label='Total Cases')
plt.plot(df_country['date'], df_country['total_deaths'], label='Total Deaths')
plt.title(f'COVID-19 in country {country}')
plt.xlabel('Date')
plt.ylabel('Count')
plt.legend()
plt.show()

## 1.5 Continental Analysis

In [ ]:
latest_date = pd.to_datetime('2022-02-14 00:00:00')
df_latest = df[df['date'] == latest_date]
df_latest['continent'].value_counts()

In [ ]:
df_latest['total_cases'].sum()

In [ ]:
continent_date = df_latest.groupby('continent')[['total_cases']].sum().reset_index()
continent_date = continent_date.dropna()

plt.figure(figsize=(12, 6))
sns.barplot(data=continent_date, x='continent', y='total_cases')
plt.title('Total COVID-19 cases by Continent')
plt.xlabel('Continent')
plt.ylabel('Total Cases')
plt.show()

---
# Урок 38: ETL, кореляції, розширені візуалізації
## Lesson 38: ETL, Correlations, Advanced Visualizations

## 2.1 Data Cleaning and Preprocessing

In [ ]:
df = pd.read_csv(data)
cols = ['new_cases', 'new_deaths', 'total_cases']
df[cols].isnull().sum()

In [ ]:
df = df.sort_values(['location', 'date'])

df['new_cases'] = df['new_cases'].fillna(0)
df['new_deaths'] = df['new_deaths'].fillna(0)
df['total_cases'] = df.groupby('location')['total_cases'].ffill()

df[cols].isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

## 2.2 Feature Engineering

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['location', 'date'])

df['growth_rate_new_cases'] = df.groupby('location')['new_cases'].pct_change()
df['growth_rate_new_deaths'] = df.groupby('location')['new_deaths'].pct_change()

## 2.3 One-Hot Encoding

In [ ]:
df = pd.get_dummies(df, columns=['location'], drop_first=True)
df

## 2.4 Save Cleaned Data

In [ ]:
df.to_csv('cleaned_covid_data.csv', index=False)

## 2.5 Correlation Analysis

In [ ]:
cols_corr = ['new_cases', 'new_deaths', 'total_cases', 'population', 'gdp_per_capita']
corr = df[cols_corr].corr()
corr

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(corr, annot=True, cmap='Blues')
plt.title('Correlation Matrix')
plt.show()

## 2.6 Distribution Visualizations

In [ ]:
df['total_cases'].hist(bins=50)
plt.title('Total Cases Distribution')
plt.show()

In [ ]:
df['total_deaths'].hist(bins=50)
plt.title('Total Deaths Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='continent', y='total_deaths_per_million', data=df)
plt.title('Deaths per million by Continent')
plt.show()

In [ ]:
sns.pairplot(df[['total_cases', 'total_deaths', 'total_vaccinations', 'population']].dropna())
plt.show()

## 2.7 Country Comparison

In [ ]:
countries_iso = ['USA', 'UKR', 'DEU']
df_countries_iso = df[df['iso_code'].isin(countries_iso)]
df_countries_iso['iso_code'].value_counts()

In [ ]:
plt.figure(figsize=(12, 6))
for country in countries_iso:
    subset = df_countries_iso[df_countries_iso['iso_code'] == country]
    plt.plot(subset['date'], subset['new_cases'], label=country)
plt.legend()
plt.title('New Cases Comparison')
plt.show()

In [ ]:
for country in countries_iso:
    subset = df[df['iso_code'] == country]
    max_cases_data = subset.loc[subset['new_cases'].idxmax(), 'date']
    min_cases_data = subset.loc[subset['new_cases'].idxmin(), 'date']
    print(f'{country}')
    print(f'Max: {max_cases_data}')
    print(f'Min: {min_cases_data}')
    print()

## 2.8 Interactive Plotly Visualization

In [ ]:
countries_5 = ['USA', 'UKR', 'DEU', 'FRA', 'ITA']
df_vacc = df[df['iso_code'].isin(countries_5)][['date', 'iso_code', 'total_vaccinations']].dropna()

In [ ]:
fig = go.Figure()

for country in countries_5:
    subset = df_vacc[df_vacc['iso_code'] == country]
    fig.add_trace(go.Scatter(x=subset['date'], y=subset['total_vaccinations'], 
                             mode='lines', name=country))

fig.update_layout(title='Vaccination Dynamics', xaxis_title='Date', yaxis_title='Total Vaccinations')
fig.show()

---
# Урок 39: Моделювання (класифікація + регресія)
## Lesson 39: Modeling (Classification + Regression)

## 3.1 Data Preparation for Modeling

In [ ]:
df = pd.read_csv(data)
df['date'] = pd.to_datetime(df['date'])

cols = ['new_cases', 'date', 'location']
df[cols].isnull().sum()

In [ ]:
df['new_cases'] = df['new_cases'].fillna(0)
df['high_cases'] = (df['new_cases'] > 1000).astype(int)

In [ ]:
df = df[['continent', 'iso_code', 'total_cases', 'total_deaths', 'total_vaccinations', 'population', 'new_cases', 'high_cases']]
df = df.dropna()
df

In [ ]:
X = df.drop(['high_cases', 'new_cases'], axis=1)
y = df['high_cases']

## 3.2 Categorical and Numerical Processing

In [ ]:
X = pd.get_dummies(X, columns=['continent'], drop_first=True)

In [ ]:
le = LabelEncoder()
X['iso_code'] = le.fit_transform(X['iso_code'])

In [ ]:
scaler = StandardScaler()
num_columns = ['total_cases', 'total_deaths', 'total_vaccinations', 'population']
X[num_columns] = scaler.fit_transform(X[num_columns])

## 3.3 Data Splitting and Balancing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=45)

In [ ]:
smote = SMOTE(random_state=45)
X_train, y_train = smote.fit_resample(X_train, y_train)

## 3.4 Classification Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier()
}

results = {}

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }
    print(f"\n{name}")
    print(confusion_matrix(y_test, y_pred))

In [ ]:
pd.DataFrame(results).T

## 3.5 Feature Selection

In [ ]:
selector = SelectKBest(score_func=f_classif, k=10)
X_new = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
selected_features

In [ ]:
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X[selected_features], y, test_size=0.2, random_state=45
)

model = RandomForestClassifier()
model.fit(X_train_new, y_train_new)

y_pred = model.predict(X_test_new)
print(f"Accuracy with selected features: {accuracy_score(y_test_new, y_pred)}")

## 3.6 Regression Models

In [ ]:
X_reg = df.drop(['new_cases', 'high_cases'], axis=1)
y_reg = df['new_cases']

X_reg = pd.get_dummies(X_reg, columns=['continent'], drop_first=True)
le_reg = LabelEncoder()
X_reg['iso_code'] = le_reg.fit_transform(X_reg['iso_code'])

scaler_reg = StandardScaler()
num_cols_reg = ['total_cases', 'total_deaths', 'total_vaccinations', 'population']
X_reg[num_cols_reg] = scaler_reg.fit_transform(X_reg[num_cols_reg])

print("X_reg shape:", X_reg.shape)
print("Features:", X_reg.columns.tolist())

In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=45
)

In [ ]:
reg_models = {
    "Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(),
    "Ridge Regression": Ridge(),
}

reg_results = {}

In [ ]:
for name, model in reg_models.items():
    model.fit(X_train_reg, y_train_reg)
    y_pred_reg = model.predict(X_test_reg)

    reg_results[name] = {
        "MSE": mean_squared_error(y_test_reg, y_pred_reg),
        "R2": r2_score(y_test_reg, y_pred_reg)
    }

pd.DataFrame(reg_results).T

## 3.7 Polynomial Regression

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_reg)

X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X_poly, y_reg, test_size=0.2, random_state=45
)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train_poly)
y_pred_poly = poly_model.predict(X_test_poly)

poly_results = {
    "Polynomial Regression (degree=2)": {
        "MSE": mean_squared_error(y_test_poly, y_pred_poly),
        "R2": r2_score(y_test_poly, y_pred_poly)
    }
}

print("Polynomial Regression Results:")
pd.DataFrame(poly_results).T

In [ ]:
all_results = {**reg_results, **poly_results}
comparison_df = pd.DataFrame(all_results).T
comparison_df = comparison_df.sort_values("R2", ascending=False)

print("All Regression Models Comparison (sorted by R²):")
comparison_df

---
# Урок 40: Оцінка, оптимізація, прогнози
## Lesson 40: Evaluation, Optimization, Forecasts

## 4.1 Model Evaluation with Confusion Matrices

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

## 4.2 False Negative Analysis

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    FN = cm[1][0]
    print(f"{name} FN: {FN}")

## 4.3 Cross-Validation

In [ ]:
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='f1')
    
    cv_results[name] = {
        "Mean F1": scores.mean(),
        "Std": scores.std()
    }

pd.DataFrame(cv_results).T

## 4.4 Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
}

In [ ]:
grid = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)
grid.fit(X_train, y_train)

In [ ]:
print(f"Best parameters: {grid.best_params_}")
best_rf = grid.best_estimator_
y_pred = best_rf.predict(X_test)
print(f"F1 with best params: {f1_score(y_test, y_pred)}")

## 4.5 Gradient Boosting Classifier

In [ ]:
gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)

y_pred = gb.predict(X_test)

gb_accuracy = accuracy_score(y_test, y_pred)
gb_f1 = f1_score(y_test, y_pred)

print(f"GB Accuracy: {gb_accuracy:.4f}")
print(f"GB F1: {gb_f1:.4f}")

## 4.6 Advanced Regression with Additional Metrics

In [ ]:
X_reg = df.drop(['new_cases', 'high_cases'], axis=1)
y = df['new_cases']

le = LabelEncoder()
X_reg['iso_code'] = le.fit_transform(X_reg['iso_code'])
X_reg['continent'] = le.fit_transform(X_reg['continent'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=45)

In [ ]:
regression_models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
}

regression_results = {}

In [ ]:
for name, model in regression_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    regression_results[name] = {
        "MSE": mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred)
    }

pd.DataFrame(regression_results).T

## 4.7 Polynomial Regression with Full Metrics

In [ ]:
lr = LinearRegression()

poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X_reg)

X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y, test_size=0.2, random_state=42
)

lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

regression_results['Polynomial'] = {
    "MSE": mean_squared_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred),
    "RMSE": root_mean_squared_error(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred)
}

pd.DataFrame(regression_results).T